# Bölüm 5c — BrickEconomy Öncelikli Örneklem Seçimi

**Amaç:** BrickEconomy günde 100 / dakikada 4 istekle sınırlı olduğu için tüm
setleri çekemiyoruz (bkz. [DATA_SOURCES.md](../DATA_SOURCES.md)). Bu not
defteri, ~2.200 setlik çekim kotasının **hangi setlere** ayrılacağını
belirliyor — henüz hiçbir BrickEconomy API çağrısı yapılmıyor, sadece
önceliklendirme mantığı ve seçilen örneklemin profili hazırlanıyor.

**Havuz:** Sadece **Brickset ile zaten eşleşmiş 6.614 set** arasından
seçim yapılıyor. Gerekçe: bu setler için retail price zaten var; BrickEconomy'den
`current_value` + `growth` + `forecast` eklemek onları hem Bölüm 2 (fiyat
tahmini) hem Faz 3 (retired-sonrası değer artışı) için **tam veri** haline
getiriyor — iki analiz aynı set üzerinde birleşiyor.

In [1]:
import pandas as pd
import numpy as np

sets_clean = pd.read_csv("../data/processed/sets_clean.csv")
brickset_prices = pd.read_csv("../data/processed/brickset_prices.csv")

merged = sets_clean.merge(brickset_prices, left_on="set_num", right_on="brickset_set_num", how="left")
ready = merged[merged["is_analysis_ready"] == True].copy()
pool = ready[ready["retail_price_us"].notna()].copy()
print(f"Havuz (Brickset ile eşleşen, is_analysis_ready): {len(pool)}")

Havuz (Brickset ile eşleşen, is_analysis_ready): 6614


## 1) "Üretim süresi" proxy metriği

**Varsayım:** LEGO, kıtlık yaratan bir lüks marka gibi davranmıyor — tam
tersine, **popüler/tercih edilen bir seti üretimde daha uzun süre tutuyor**,
az satan bir seti ise daha erken raftan kaldırıyor. Bu nedenle bir setin
`date_first_available`'dan `date_last_available`'a kadar geçen süresi
("üretim süresi"), o setin ne kadar tercih edildiğine dair makul bir proxy
olarak kullanılabilir.

Bu, Faz 4'te tanımlanan "üretim süresi" proxy metriğiyle **kavramsal olarak
aynı** — burada BrickEconomy önceliklendirmesi için, orada tema
ömrü/başarı analizinde kullanılabilir; ikisi çapraz besleniyor.

In [2]:
pool["date_first_available"] = pd.to_datetime(pool["date_first_available"], errors="coerce", utc=True)
pool["date_last_available"] = pd.to_datetime(pool["date_last_available"], errors="coerce", utc=True)
pool["production_days"] = (pool["date_last_available"] - pool["date_first_available"]).dt.days

has_days = pool["production_days"].notna()
print(f"Üretim süresi hesaplanabilen: {has_days.sum()} / {len(pool)}")
print(f"Eksik (en sona atılacak): {(~has_days).sum()} / {len(pool)}")

neg = pool[pool["production_days"] < 0]
print(f"Negatif üretim süresi (veri hatası şüphesi): {len(neg)}")

Üretim süresi hesaplanabilen: 6149 / 6614
Eksik (en sona atılacak): 465 / 6614
Negatif üretim süresi (veri hatası şüphesi): 0


## 2) Bileşik öncelik skoru

`combined_score = z(production_days) + z(num_parts)` — ikisi de standardize edilip toplanıyor (basit, yorumlanabilir). Üretim süresi eksik olan setler skordan bağımsız olarak listenin en sonuna atılıyor.

In [3]:
z_days = (pool.loc[has_days, "production_days"] - pool.loc[has_days, "production_days"].mean()) / pool.loc[has_days, "production_days"].std()
pool["z_production_days"] = np.nan
pool.loc[has_days, "z_production_days"] = z_days
pool["z_num_parts"] = (pool["num_parts"] - pool["num_parts"].mean()) / pool["num_parts"].std()
pool["combined_score"] = pool["z_production_days"] + pool["z_num_parts"]
pool["has_days"] = has_days

# has_days=True önce (True > False sıralamada), sonra combined_score azalan,
# eksik grup içinde tiebreak olarak num_parts azalan
priority_order = pool.sort_values(by=["has_days", "combined_score", "num_parts"], ascending=[False, False, False])
top2200 = priority_order.head(2200)

print(f"Seçilen 2.200 içinde üretim-süresi-dolu olan: {top2200['has_days'].sum()}")
print(f"Seçilen 2.200 içinde eksik (sona atılmış ama yine de kotaya giren): {(~top2200['has_days']).sum()}")
print()
print(f"Medyan num_parts: {top2200['num_parts'].median():.0f}")
print(f"Medyan üretim süresi: {top2200.loc[top2200['has_days'], 'production_days'].median():.0f} gün (~{top2200.loc[top2200['has_days'], 'production_days'].median()/30:.1f} ay)")
print(f"Medyan yıl: {top2200['year'].median():.0f}")

Seçilen 2.200 içinde üretim-süresi-dolu olan: 2200
Seçilen 2.200 içinde eksik (sona atılmış ama yine de kotaya giren): 0

Medyan num_parts: 597
Medyan üretim süresi: 742 gün (~24.7 ay)
Medyan yıl: 2019


## 3) Seçilen 2.200 setin tema dağılımı

In [4]:
top_themes = top2200["theme_name"].value_counts()
print(f"Seçilen 2.200 set, {top2200['theme_name'].nunique()} farklı temaya yayılıyor.")
print()
print("En sık 15 tema:")
print(top_themes.head(15))

Seçilen 2.200 set, 165 farklı temaya yayılıyor.

En sık 15 tema:
theme_name
Star Wars                195
Technic                  148
Friends                  122
Creator 3-in-1           114
Ninjago                   91
Minecraft                 69
Harry Potter              58
Police                    51
Classic                   43
Town                      42
LEGO Ideas and CUUSOO     41
Architecture              38
Monkie Kid                38
City                      37
Speed Champions           37
Name: count, dtype: int64


**Gözlem:** Bu bileşik önceliklendirme (üretim süresi + parça sayısı) ile en
çok öne çıkan temalar **Star Wars (195), Technic (148), Friends (122),
Creator 3-in-1 (114) ve Ninjago (91)** oluyor — bunları Minecraft, Harry
Potter, Police, Classic, Town gibi temalar takip ediyor. Toplamda 165 farklı
temaya yayılmış durumda (tek birkaç temaya aşırı yığılma yok). Bu, "en
revaçta temalar hangileri" sorusuna erken/kaba bir cevap veriyor — hem uzun
üretimde kalmış hem büyük setler üreten temalar bunlar. Bölüm 1/4'teki tema
başarı analizinde bu sıralamanın tutarlı çıkıp çıkmadığı ayrıca
doğrulanabilir.

## 4) Minifigür kotası (~800) — BrickEconomy endpoint durumu

**BrickEconomy'nin ayrı bir minifigür endpoint'i VAR:** `GET
/minifig/{minifigNumber}` (ör. `"sw0509"`), `current_value_new` ve
`price_events_new` döndürüyor (OpenAPI şemasından doğrulandı).

**Ancak format uyuşmazlığı var:** BrickEconomy'nin minifig kodları
(tema-önekli, `"twn402"`, `"sw0509"` gibi) Rebrickable'ın `fig_num`
formatıyla (`"fig-000001"`) uyumlu değil, doğrudan bir dönüşüm yok.

**Önceliklendirme için — henüz karar verilmedi, seçenekler:**

1. **Set-yanıtlarından türetme:** Yukarıdaki 2.200 setin BrickEconomy
   yanıtlarındaki `minifigs: [...]` alanlarından toplanan benzersiz kodlar
   (doğru format garanti, zaten çekilen setlerle ilişkili minifigürler
   önceliklenir).
2. **Rastgele örneklem:** rpmoo ve Brickset'te minifig-bazlı fiyat/nadirlik
   verisi olmadığı doğrulandı (rpmoo sadece set fiyatı çekmiş; Brickset'in
   `getSets` çıktısı sadece minifig *kodlarını* veriyor, fiyat değil) —
   yani "en pahalı/nadir" diye önceliklendirecek bağımsız bir sinyal şu an
   elimizde yok. Bu durumda Rebrickable'ın `minifigs.csv`'sindeki mevcut
   minifigürlerden rastgele bir örneklemle başlamak tek pratik alternatif.

Bu iki seçenek arasında karar **kullanıcıya bırakıldı** — henüz uygulanmadı.